# Disentangling Length Effect and Hydrogen-Bond Network Effect

This notebook tests whether hydrogen-bond descriptors explain mechanical properties beyond the trivial effect that longer proteins have more residues and more possible hydrogen bonds.

In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-mprl")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, ttest_1samp, wilcoxon
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", context="talk")

BASE_DIR = Path("outputs/hbond_analysis")
PLOTS_DIR = BASE_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_PATH = BASE_DIR / "hbond_analysis_table.csv"
DETAIL_PATH = BASE_DIR / "hbond_details.parquet"
MD_PATH = Path("hbond_analysis.md")
RANDOM_STATE = 7

print(FEATURE_PATH.resolve())
print(DETAIL_PATH.resolve())

/mnt/nas/jianquanzhao/gits/Mechanical-protein-RL/model/reward_module/mechanical-properties-predictor/outputs/hbond_analysis/hbond_analysis_table.csv
/mnt/nas/jianquanzhao/gits/Mechanical-protein-RL/model/reward_module/mechanical-properties-predictor/outputs/hbond_analysis/hbond_details.parquet


In [2]:
analysis_df = pd.read_csv(FEATURE_PATH)
details_df = pd.read_parquet(DETAIL_PATH)
analysis_df["log_sequence_length"] = np.log1p(analysis_df["sequence_length"].astype(float))
analysis_df["log1p_v127"] = np.log1p(analysis_df["v127"].astype(float))
analysis_df["log1p_v128"] = np.log1p(analysis_df["v128"].astype(float))
print("analysis", analysis_df.shape, "details", details_df.shape)
analysis_df.head()

analysis (7041, 79) details (497223, 17)


,PDB_ID,v127,v128,Sequence,pdb_path,sequence_length,parse_ok,atom_count,residue_count,hbond_count,...,seq_class_nonlocal_fraction,seq_class_interchain_count,seq_class_interchain_per_residue,seq_class_interchain_fraction,seq_class_same_residue_count,seq_class_same_residue_per_residue,seq_class_same_residue_fraction,log1p_v127,log1p_v128,log_sequence_length
0,2w1nA02,320.23,395.1016,DFKASEINKKNVTVTWTEPETTEGLEGYILYKDGKKVAEIGKDETS...,/home/jianquanzhao/data/tsinghua/mpprd/pdbs/pd...,81.0,1.0,1304.0,81.0,54.0,...,0.796296,0.0,0.0,0.0,0.0,0.0,0.0,5.772157,5.981671,4.406719
1,2jsnA00,354.20,434.8929,MGLDSYAPRAEAEKTFSYPLDLLLKLDERVLVAFGQRDGIRVGAVL...,/home/jianquanzhao/data/tsinghua/mpprd/pdbs/pd...,88.0,1.0,1371.0,88.0,63.0,...,0.555556,0.0,0.0,0.0,0.0,0.0,0.0,5.872681,6.077397,4.488636
2,2dadA00,289.64,467.7182,GSSGSSGQILFSEPQFQGSQSFEETTSQIDDSFSTKSCRVSGGSWV...,/home/jianquanzhao/data/tsinghua/mpprd/pdbs/pd...,90.0,1.0,1274.0,90.0,41.0,...,0.682927,0.0,0.0,0.0,0.0,0.0,0.0,5.672085,6.150002,4.510860
3,1ueyA00,327.69,528.9447,GSSGSSGPTPAPVYDVPNPPFDLELTDQLDKSVQLSWTPGDDNNSP...,/home/jianquanzhao/data/tsinghua/mpprd/pdbs/pd...,124.0,1.0,1823.0,124.0,56.0,...,0.732143,0.0,0.0,0.0,0.0,0.0,0.0,5.795115,6.272773,4.828314
4,2dm3A00,310.61,497.6552,GSSGSSGFRPFLQAPGDLTVQEGKLCRMDCKVSGLPTPDLSWQLDG...,/home/jianquanzhao/data/tsinghua/mpprd/pdbs/pd...,107.0,1.0,1570.0,107.0,35.0,...,0.714286,0.0,0.0,0.0,0.0,0.0,0.0,5.741752,6.211915,4.682131


In [3]:
length_map = analysis_df.set_index("PDB_ID")["sequence_length"].astype(float)
details = details_df.copy()
details["sequence_length"] = details["PDB_ID"].map(length_map)
details["seq_sep_positive"] = details["seq_sep"].where(details["seq_sep"] >= 0, np.nan)
details["seq_sep_norm"] = details["seq_sep_positive"] / details["sequence_length"].replace(0, np.nan)
details["is_long_range"] = details["seq_sep_positive"] >= 12
details["is_very_long_range"] = details["seq_sep_positive"] >= 24
details["is_nonlocal_backbone_backbone"] = (details["seq_class"] == "nonlocal") & (details["role_type"] == "backbone_to_backbone")
details["is_strong"] = (details["d_a_distance"] <= 3.0) & (details["dha_angle"] >= 150.0)
details["is_strong_nonlocal"] = details["is_strong"] & (details["seq_class"] == "nonlocal")

topology_rows = []
for pdb_id, group in details.groupby("PDB_ID"):
    length = float(length_map.loc[pdb_id])
    n = max(1, len(group))
    topology_rows.append({
        "PDB_ID": pdb_id,
        "mean_seq_sep": float(group["seq_sep_positive"].mean()),
        "median_seq_sep": float(group["seq_sep_positive"].median()),
        "max_seq_sep": float(group["seq_sep_positive"].max()),
        "hbond_contact_order": float(group["seq_sep_norm"].mean()),
        "long_range_hbond_count": float(group["is_long_range"].sum()),
        "long_range_hbond_per_residue": float(group["is_long_range"].sum() / max(1.0, length)),
        "long_range_hbond_fraction": float(group["is_long_range"].mean()),
        "very_long_range_hbond_count": float(group["is_very_long_range"].sum()),
        "very_long_range_hbond_per_residue": float(group["is_very_long_range"].sum() / max(1.0, length)),
        "very_long_range_hbond_fraction": float(group["is_very_long_range"].mean()),
        "nonlocal_backbone_backbone_count": float(group["is_nonlocal_backbone_backbone"].sum()),
        "nonlocal_backbone_backbone_per_residue": float(group["is_nonlocal_backbone_backbone"].sum() / max(1.0, length)),
        "nonlocal_backbone_backbone_fraction": float(group["is_nonlocal_backbone_backbone"].mean()),
        "strong_nonlocal_count": float(group["is_strong_nonlocal"].sum()),
        "strong_nonlocal_per_residue": float(group["is_strong_nonlocal"].sum() / max(1.0, length)),
        "strong_nonlocal_fraction": float(group["is_strong_nonlocal"].mean()),
    })

topology_df = pd.DataFrame(topology_rows)
analysis_df = analysis_df.merge(topology_df, on="PDB_ID", how="left")
topology_cols = [col for col in topology_df.columns if col != "PDB_ID"]
analysis_df[topology_cols] = analysis_df[topology_cols].fillna(0.0)
analysis_df.to_csv(BASE_DIR / "hbond_length_disentanglement_table.csv", index=False)
topology_df.to_csv(BASE_DIR / "hbond_topology_features.csv", index=False)
analysis_df[topology_cols].describe().T.head(20)

,count,mean,std,min,25%,50%,75%,max
mean_seq_sep,7041.0,20.630234,11.584731,2.571429,12.100000,18.310345,26.000000,111.455150
median_seq_sep,7041.0,13.746130,8.322588,2.000000,8.000000,12.000000,17.000000,72.500000
max_seq_sep,7041.0,78.669933,65.212682,5.000000,36.000000,62.000000,96.000000,542.000000
hbond_contact_order,7041.0,0.217417,0.079457,0.030879,0.164536,0.214198,0.264423,0.767380
long_range_hbond_count,7041.0,38.395540,38.903752,0.000000,13.000000,28.000000,49.000000,352.000000
long_range_hbond_per_residue,7041.0,0.312661,0.138614,0.000000,0.211356,0.314607,0.410256,0.945946
long_range_hbond_fraction,7041.0,0.496562,0.156649,0.000000,0.400000,0.510949,0.605263,1.000000
very_long_range_hbond_count,7041.0,23.604744,28.165927,0.000000,4.000000,15.000000,32.000000,286.000000
very_long_range_hbond_per_residue,7041.0,0.177924,0.130302,0.000000,0.067568,0.166667,0.274725,0.706827
very_long_range_hbond_fraction,7041.0,0.272005,0.178514,0.000000,0.126761,0.275862,0.408163,1.000000


In [4]:
def corr_pair(x, y):
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 5 or np.unique(x[valid]).size <= 1 or np.unique(y[valid]).size <= 1:
        return np.nan, np.nan, np.nan, np.nan
    pr, pp = pearsonr(x[valid], y[valid])
    sr, sp = spearmanr(x[valid], y[valid])
    return float(pr), float(pp), float(sr), float(sp)


def residualize(values, length_values):
    x = np.asarray(length_values, dtype=float).reshape(-1, 1)
    y = np.asarray(values, dtype=float)
    model = LinearRegression().fit(x, y)
    return y - model.predict(x)


target_cols = ["log1p_v127", "log1p_v128"]
candidate_cols = [
    "hbond_count", "hbond_per_residue", "chem_type_N_to_O_count", "chem_type_N_to_O_per_residue",
    "seq_class_nonlocal_count", "seq_class_nonlocal_per_residue", "seq_class_nonlocal_fraction",
    "role_type_backbone_to_backbone_count", "role_type_backbone_to_backbone_per_residue", "role_type_backbone_to_backbone_fraction",
    "strong_hbond_fraction", "weak_hbond_fraction", "d_a_distance_mean", "h_a_distance_mean", "dha_angle_mean",
    "hbond_contact_order", "long_range_hbond_per_residue", "long_range_hbond_fraction",
    "nonlocal_backbone_backbone_per_residue", "nonlocal_backbone_backbone_fraction",
    "strong_nonlocal_per_residue", "strong_nonlocal_fraction",
]

corr_rows = []
partial_rows = []
length_values = analysis_df["log_sequence_length"].to_numpy(dtype=float)
for feature in candidate_cols:
    x_raw = analysis_df[feature].to_numpy(dtype=float)
    x_resid = residualize(x_raw, length_values)
    for target in target_cols:
        y_raw = analysis_df[target].to_numpy(dtype=float)
        y_resid = residualize(y_raw, length_values)
        pr, pp, sr, sp = corr_pair(x_raw, y_raw)
        corr_rows.append({"feature": feature, "target": target, "pearson": pr, "pearson_p": pp, "spearman": sr, "spearman_p": sp})
        ppr, ppp, psr, psp = corr_pair(x_resid, y_resid)
        partial_rows.append({"feature": feature, "target": target, "partial_pearson": ppr, "partial_pearson_p": ppp, "partial_spearman": psr, "partial_spearman_p": psp})

corr_df = pd.DataFrame(corr_rows)
partial_df = pd.DataFrame(partial_rows)
joined_corr = corr_df.merge(partial_df, on=["feature", "target"])
joined_corr["abs_partial_spearman"] = joined_corr["partial_spearman"].abs()
joined_corr["spearman_drop_after_length_control"] = joined_corr["spearman"] - joined_corr["partial_spearman"]
joined_corr.to_csv(BASE_DIR / "hbond_length_controlled_partial_correlations.csv", index=False)
display(joined_corr.sort_values(["target", "abs_partial_spearman"], ascending=[True, False]).groupby("target").head(10))

,feature,target,pearson,pearson_p,spearman,spearman_p,partial_pearson,partial_pearson_p,partial_spearman,partial_spearman_p,abs_partial_spearman,spearman_drop_after_length_control
40,strong_nonlocal_per_residue,log1p_v127,0.654840,0.000000e+00,0.682213,0.000000e+00,0.513555,0.000000e+00,0.524571,0.000000e+00,0.524571,0.157642
6,chem_type_N_to_O_per_residue,log1p_v127,0.710910,0.000000e+00,0.724466,0.000000e+00,0.487759,0.000000e+00,0.494053,0.000000e+00,0.494053,0.230413
2,hbond_per_residue,log1p_v127,0.715951,0.000000e+00,0.730404,0.000000e+00,0.483112,0.000000e+00,0.490546,0.000000e+00,0.490546,0.239858
10,seq_class_nonlocal_per_residue,log1p_v127,0.705227,0.000000e+00,0.724490,0.000000e+00,0.458486,0.000000e+00,0.476161,0.000000e+00,0.476161,0.248329
16,role_type_backbone_to_backbone_per_residue,log1p_v127,0.633377,0.000000e+00,0.624481,0.000000e+00,0.432733,2.637718e-319,0.454435,0.000000e+00,0.454435,0.170046
32,long_range_hbond_per_residue,log1p_v127,0.710450,0.000000e+00,0.740817,0.000000e+00,0.425115,4.671606e-307,0.449745,0.000000e+00,0.449745,0.291072
36,nonlocal_backbone_backbone_per_residue,log1p_v127,0.582944,0.000000e+00,0.581841,0.000000e+00,0.386450,1.445756e-249,0.411349,1.038850e-285,0.411349,0.170492
42,strong_nonlocal_fraction,log1p_v127,0.357812,1.046308e-211,0.378803,4.451668e-239,0.381228,2.252288e-242,0.389334,1.353514e-253,0.389334,-0.010532
28,dha_angle_mean,log1p_v127,0.294848,3.114048e-141,0.278724,8.348255e-126,0.401043,2.263249e-270,0.374985,6.026417e-234,0.374985,-0.096261
20,strong_hbond_fraction,log1p_v127,0.295205,1.381519e-141,0.297939,2.619968e-144,0.380036,9.472009e-241,0.372072,4.451279e-230,0.372072,-0.074133


In [5]:
analysis_df["length_bin"] = pd.qcut(analysis_df["sequence_length"], q=4, labels=["Q1_short", "Q2", "Q3", "Q4_long"])
strat_rows = []
strat_features = ["hbond_per_residue", "seq_class_nonlocal_per_residue", "hbond_contact_order", "nonlocal_backbone_backbone_per_residue", "strong_nonlocal_fraction"]
for bin_name, group in analysis_df.groupby("length_bin", observed=True):
    for feature in strat_features:
        for target in target_cols:
            pr, pp, sr, sp = corr_pair(group[feature].to_numpy(float), group[target].to_numpy(float))
            strat_rows.append({
                "length_bin": str(bin_name),
                "n": int(len(group)),
                "length_min": float(group["sequence_length"].min()),
                "length_max": float(group["sequence_length"].max()),
                "feature": feature,
                "target": target,
                "pearson": pr,
                "spearman": sr,
            })
strat_df = pd.DataFrame(strat_rows)
strat_df.to_csv(BASE_DIR / "hbond_length_stratified_correlations.csv", index=False)
display(strat_df.sort_values(["target", "feature", "length_bin"]))

,length_bin,n,length_min,length_max,feature,target,pearson,spearman
4,Q1_short,1809,27.0,58.0,hbond_contact_order,log1p_v127,0.019678,0.046510
14,Q2,1742,59.0,89.0,hbond_contact_order,log1p_v127,0.245626,0.283404
24,Q3,1751,90.0,125.0,hbond_contact_order,log1p_v127,0.198375,0.217905
34,Q4_long,1739,126.0,586.0,hbond_contact_order,log1p_v127,0.017543,0.029101
0,Q1_short,1809,27.0,58.0,hbond_per_residue,log1p_v127,0.579773,0.558991
10,Q2,1742,59.0,89.0,hbond_per_residue,log1p_v127,0.548287,0.578079
20,Q3,1751,90.0,125.0,hbond_per_residue,log1p_v127,0.498400,0.475630
30,Q4_long,1739,126.0,586.0,hbond_per_residue,log1p_v127,0.425448,0.409028
6,Q1_short,1809,27.0,58.0,nonlocal_backbone_backbone_per_residue,log1p_v127,0.486276,0.462082
16,Q2,1742,59.0,89.0,nonlocal_backbone_backbone_per_residue,log1p_v127,0.466555,0.493689


In [6]:
match_features = ["hbond_per_residue", "seq_class_nonlocal_per_residue", "hbond_contact_order", "nonlocal_backbone_backbone_per_residue", "strong_nonlocal_fraction"]

def matched_length_differences(df, target, feature_cols, top_fraction=0.10, max_length_diff=5):
    high_cut = df[target].quantile(1.0 - top_fraction)
    low_cut = df[target].quantile(top_fraction)
    high = df[df[target] >= high_cut].sort_values(target, ascending=False).copy()
    low = df[df[target] <= low_cut].copy()
    used = set()
    rows = []
    for _, hi in high.iterrows():
        candidates = low.loc[~low.index.isin(used)].copy()
        if candidates.empty:
            break
        candidates["length_diff"] = (candidates["sequence_length"] - hi["sequence_length"]).abs()
        candidates = candidates[candidates["length_diff"] <= max_length_diff]
        if candidates.empty:
            continue
        lo = candidates.sort_values(["length_diff", target], ascending=[True, True]).iloc[0]
        used.add(lo.name)
        row = {
            "target": target,
            "high_PDB_ID": hi["PDB_ID"],
            "low_PDB_ID": lo["PDB_ID"],
            "high_length": float(hi["sequence_length"]),
            "low_length": float(lo["sequence_length"]),
            "length_diff": float(abs(hi["sequence_length"] - lo["sequence_length"])),
            "high_target": float(hi[target]),
            "low_target": float(lo[target]),
        }
        for feature in feature_cols:
            row[f"delta_{feature}"] = float(hi[feature] - lo[feature])
        rows.append(row)
    return pd.DataFrame(rows)

matched_frames = [matched_length_differences(analysis_df, target, match_features) for target in ["v127", "v128"]]
matched_df = pd.concat(matched_frames, ignore_index=True)
matched_df.to_csv(BASE_DIR / "hbond_matched_length_pairs.csv", index=False)

matched_summary_rows = []
for target, group in matched_df.groupby("target"):
    for feature in match_features:
        values = group[f"delta_{feature}"].dropna().to_numpy(float)
        if len(values) == 0:
            continue
        stat, pvalue = ttest_1samp(values, popmean=0.0)
        matched_summary_rows.append({
            "target": target,
            "feature": feature,
            "n_pairs": int(len(values)),
            "mean_high_minus_low": float(np.mean(values)),
            "median_high_minus_low": float(np.median(values)),
            "ttest_p": float(pvalue),
        })
matched_summary = pd.DataFrame(matched_summary_rows).sort_values(["target", "mean_high_minus_low"], ascending=[True, False])
matched_summary.to_csv(BASE_DIR / "hbond_matched_length_pair_summary.csv", index=False)
display(matched_summary)

,target,feature,n_pairs,mean_high_minus_low,median_high_minus_low,ttest_p
4,v127,strong_nonlocal_fraction,38,0.215076,0.256759,1.242815e-09
0,v127,hbond_per_residue,38,0.184219,0.202587,4.327910e-06
1,v127,seq_class_nonlocal_per_residue,38,0.149166,0.159784,1.707004e-05
3,v127,nonlocal_backbone_backbone_per_residue,38,0.080326,0.092172,6.247785e-04
2,v127,hbond_contact_order,38,0.022169,0.014713,1.313284e-01
7,v128,hbond_contact_order,36,0.010398,0.017630,4.764630e-01
9,v128,strong_nonlocal_fraction,36,-0.004616,0.010995,8.850020e-01
8,v128,nonlocal_backbone_backbone_per_residue,36,-0.063245,-0.063636,4.443030e-03
6,v128,seq_class_nonlocal_per_residue,36,-0.065361,-0.041113,2.484974e-02
5,v128,hbond_per_residue,36,-0.088402,-0.047533,1.194015e-02


In [7]:
all_count_cols = [col for col in analysis_df.columns if col.endswith("_count") or col == "hbond_count"]
raw_count_cols = [col for col in all_count_cols if analysis_df[col].nunique(dropna=True) > 1]
density_cols = [col for col in analysis_df.columns if col.endswith("_per_residue") or col.endswith("_fraction")]
geometry_cols = ["d_a_distance_mean", "d_a_distance_std", "h_a_distance_mean", "h_a_distance_std", "dha_angle_mean", "dha_angle_std", "strong_hbond_fraction", "weak_hbond_fraction"]
topology_model_cols = ["hbond_contact_order", "mean_seq_sep", "long_range_hbond_per_residue", "long_range_hbond_fraction", "very_long_range_hbond_per_residue", "nonlocal_backbone_backbone_per_residue", "nonlocal_backbone_backbone_fraction", "strong_nonlocal_per_residue", "strong_nonlocal_fraction"]

model_base = analysis_df.copy()
train_idx, temp_idx = train_test_split(model_base.index, test_size=0.2, random_state=RANDOM_STATE)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=RANDOM_STATE)

def add_train_residual_features(df, train_indices, feature_cols):
    out = df.copy()
    x_train = np.log1p(out.loc[train_indices, "sequence_length"].to_numpy(float)).reshape(-1, 1)
    x_all = np.log1p(out["sequence_length"].to_numpy(float)).reshape(-1, 1)
    residual_cols = []
    for col in feature_cols:
        y_train = np.log1p(out.loc[train_indices, col].clip(lower=0).to_numpy(float))
        y_all = np.log1p(out[col].clip(lower=0).to_numpy(float))
        reg = LinearRegression().fit(x_train, y_train)
        residual_col = f"resid_log1p_{col}"
        out[residual_col] = y_all - reg.predict(x_all)
        residual_cols.append(residual_col)
    return out, residual_cols

model_base, residual_cols = add_train_residual_features(model_base, train_idx, raw_count_cols)

feature_sets = {
    "length_only": ["sequence_length"],
    "length_plus_density_geometry": ["sequence_length"] + density_cols + geometry_cols,
    "length_plus_residual_counts": ["sequence_length"] + residual_cols + geometry_cols,
    "length_plus_topology": ["sequence_length"] + topology_model_cols + geometry_cols,
    "length_plus_all_hbond_controlled": ["sequence_length"] + density_cols + residual_cols + topology_model_cols + geometry_cols,
}
feature_sets = {name: sorted(set(cols)) for name, cols in feature_sets.items()}

def evaluate_model_set(name, cols):
    cols = [col for col in cols if col in model_base.columns and model_base[col].nunique(dropna=True) > 1]
    X = model_base[cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    y_log = np.log1p(model_base[["v127", "v128"]].to_numpy(float))
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("rf", MultiOutputRegressor(RandomForestRegressor(n_estimators=300, min_samples_leaf=3, max_features="sqrt", random_state=RANDOM_STATE, n_jobs=-1))),
    ])
    model.fit(X.loc[train_idx], y_log[model_base.index.get_indexer(train_idx)])
    rows = []
    for split_name, indices in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
        pred = np.expm1(model.predict(X.loc[indices]))
        true = model_base.loc[indices, ["v127", "v128"]].to_numpy(float)
        row = {"model": name, "split": split_name, "n_features": len(cols), "n": int(len(indices))}
        for j, target_name in enumerate(["toughness_v127", "strength_v128"]):
            row[f"{target_name}/r2"] = float(r2_score(true[:, j], pred[:, j]))
            row[f"{target_name}/mae"] = float(mean_absolute_error(true[:, j], pred[:, j]))
            row[f"{target_name}/rmse"] = float(np.sqrt(mean_squared_error(true[:, j], pred[:, j])))
            row[f"{target_name}/spearman"] = float(spearmanr(true[:, j], pred[:, j]).correlation)
        rows.append(row)
    return rows

model_rows = []
for name, cols in feature_sets.items():
    print("training", name, len(cols))
    model_rows.extend(evaluate_model_set(name, cols))
model_metrics = pd.DataFrame(model_rows)
model_metrics.to_csv(BASE_DIR / "hbond_length_controlled_model_ablation_metrics.csv", index=False)
display(model_metrics[model_metrics["split"] == "test"].sort_values("model"))

training length_only 1
training length_plus_density_geometry 52
training length_plus_residual_counts 28
training length_plus_topology 18
training length_plus_all_hbond_controlled 73


,model,split,n_features,n,toughness_v127/r2,toughness_v127/mae,toughness_v127/rmse,toughness_v127/spearman,strength_v128/r2,strength_v128/mae,strength_v128/rmse,strength_v128/spearman
2,length_only,test,1,705,0.537987,47.630817,73.610147,0.819188,0.216211,67.976608,191.534101,0.836440
14,length_plus_all_hbond_controlled,test,63,705,0.687562,36.154843,60.533024,0.902092,0.342895,64.481977,175.373502,0.845386
5,length_plus_density_geometry,test,42,705,0.665457,37.449776,62.637775,0.894252,0.312582,63.535798,179.373020,0.839316
8,length_plus_residual_counts,test,28,705,0.678058,36.553541,61.446848,0.900801,0.324674,64.840546,177.788337,0.845157
11,length_plus_topology,test,18,705,0.678758,36.695767,61.379956,0.900551,0.327551,63.941835,177.409277,0.844270


In [8]:
test_metrics = model_metrics[model_metrics["split"] == "test"].copy()
base = test_metrics[test_metrics["model"] == "length_only"].iloc[0]
for metric in ["toughness_v127/r2", "strength_v128/r2", "toughness_v127/spearman", "strength_v128/spearman"]:
    test_metrics[f"delta_{metric}"] = test_metrics[metric] - float(base[metric])
test_metrics.to_csv(BASE_DIR / "hbond_length_controlled_model_ablation_test_deltas.csv", index=False)

plot_df = test_metrics.melt(id_vars=["model"], value_vars=["toughness_v127/r2", "strength_v128/r2"], var_name="target_metric", value_name="r2")
plt.figure(figsize=(12, 6))
sns.barplot(data=plot_df, x="model", y="r2", hue="target_metric")
plt.xticks(rotation=25, ha="right")
plt.title("Length-controlled hydrogen-bond feature ablation, test R2")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "hbond_length_controlled_model_ablation_r2.png", dpi=220)
plt.close()

partial_plot = joined_corr[joined_corr["target"].isin(target_cols)].copy()
partial_plot = partial_plot.sort_values("abs_partial_spearman", ascending=False).head(20)
plt.figure(figsize=(10, 8))
sns.barplot(data=partial_plot, x="partial_spearman", y="feature", hue="target")
plt.axvline(0, color="black", linewidth=1)
plt.title("Top partial Spearman correlations after length control")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "hbond_partial_correlation_after_length_control.png", dpi=220)
plt.close()

display(test_metrics)

,model,split,n_features,n,toughness_v127/r2,toughness_v127/mae,toughness_v127/rmse,toughness_v127/spearman,strength_v128/r2,strength_v128/mae,strength_v128/rmse,strength_v128/spearman,delta_toughness_v127/r2,delta_strength_v128/r2,delta_toughness_v127/spearman,delta_strength_v128/spearman
2,length_only,test,1,705,0.537987,47.630817,73.610147,0.819188,0.216211,67.976608,191.534101,0.836440,0.000000,0.000000,0.000000,0.000000
5,length_plus_density_geometry,test,42,705,0.665457,37.449776,62.637775,0.894252,0.312582,63.535798,179.373020,0.839316,0.127470,0.096370,0.075063,0.002876
8,length_plus_residual_counts,test,28,705,0.678058,36.553541,61.446848,0.900801,0.324674,64.840546,177.788337,0.845157,0.140071,0.108463,0.081612,0.008717
11,length_plus_topology,test,18,705,0.678758,36.695767,61.379956,0.900551,0.327551,63.941835,177.409277,0.844270,0.140771,0.111340,0.081363,0.007830
14,length_plus_all_hbond_controlled,test,63,705,0.687562,36.154843,60.533024,0.902092,0.342895,64.481977,175.373502,0.845386,0.149575,0.126684,0.082904,0.008946


In [9]:
def markdown_table(frame, digits=4):
    out = frame.copy()
    for col in out.columns:
        if pd.api.types.is_numeric_dtype(out[col]):
            out[col] = out[col].map(lambda x: f"{x:.{digits}f}" if pd.notna(x) else "nan")
    return out.to_markdown(index=False)

top_partial_v127 = joined_corr[joined_corr["target"] == "log1p_v127"].sort_values("abs_partial_spearman", ascending=False).head(8)
top_partial_v128 = joined_corr[joined_corr["target"] == "log1p_v128"].sort_values("abs_partial_spearman", ascending=False).head(8)
strat_focus = strat_df[strat_df["feature"].isin(["hbond_per_residue", "seq_class_nonlocal_per_residue", "hbond_contact_order"])].copy()
match_focus = matched_summary.sort_values(["target", "mean_high_minus_low"], ascending=[True, False]).groupby("target").head(5)

section = f"""

## Length-Controlled Hydrogen-Bond Analysis

### Hypothesis

Raw hydrogen-bond counts are strongly correlated with mechanical properties, but part of this signal may be caused by protein length: longer proteins have more atoms, more donor/acceptor pairs, and therefore more hydrogen bonds. The hypothesis tested here is:

```text
Hydrogen-bond network descriptors still explain toughness and strength after controlling for sequence length.
```

### Method

The analysis used the cached hydrogen-bond table from `outputs/hbond_analysis/` and did not rescan PDB files. Five complementary tests were performed:

1. **Length-normalized descriptors**: counts were converted to per-residue or fraction features.
2. **Residualized descriptors**: raw count features were regressed against `log1p(sequence_length)`, and residuals were used as length-independent hydrogen-bond signals.
3. **Partial correlation**: both hydrogen-bond features and mechanical labels were residualized against `log1p(sequence_length)`, then correlated.
4. **Length-stratified analysis**: proteins were split into length quartiles and correlations were recomputed within each quartile.
5. **Model ablation**: random-forest regressors were trained with increasing feature groups: length only, length plus density/geometry, length plus residual counts, length plus topology, and length plus all controlled hydrogen-bond features.

Topology-aware features were also added from hydrogen-bond pair details, including `hbond_contact_order`, long-range hydrogen-bond fraction, nonlocal backbone-backbone hydrogen bonds, and strong nonlocal hydrogen bonds.

### Partial Correlation Results

After controlling for length, the strongest remaining hydrogen-bond associations are much smaller than the raw correlations. This is expected and biologically important: a large part of the raw signal was indeed a length/size signal.

#### Top Partial Correlations With log1p(v127), Toughness

{markdown_table(top_partial_v127[['feature', 'spearman', 'partial_spearman', 'spearman_drop_after_length_control']], 5)}

#### Top Partial Correlations With log1p(v128), Strength

{markdown_table(top_partial_v128[['feature', 'spearman', 'partial_spearman', 'spearman_drop_after_length_control']], 5)}

Full table: `outputs/hbond_analysis/hbond_length_controlled_partial_correlations.csv`.

### Length-Stratified Results

The table below shows representative within-bin Spearman correlations. These values ask whether hydrogen-bond density/topology still matters among proteins of similar length.

{markdown_table(strat_focus[['length_bin', 'n', 'length_min', 'length_max', 'feature', 'target', 'spearman']], 4)}

Full table: `outputs/hbond_analysis/hbond_length_stratified_correlations.csv`.

### Matched-Length Pair Results

High-performance proteins were matched to low-performance proteins with sequence lengths within 5 residues. The table reports high-minus-low feature differences. Positive values mean the high-performance member has more of that hydrogen-bond descriptor despite comparable length.

{markdown_table(match_focus[['target', 'feature', 'n_pairs', 'mean_high_minus_low', 'median_high_minus_low', 'ttest_p']], 5)}

Full matched-pair table: `outputs/hbond_analysis/hbond_matched_length_pairs.csv`.

### Model Ablation Results

The key test is whether hydrogen-bond descriptors improve test performance beyond a length-only model.

{markdown_table(test_metrics[['model', 'n_features', 'toughness_v127/r2', 'strength_v128/r2', 'toughness_v127/spearman', 'strength_v128/spearman', 'delta_toughness_v127/r2', 'delta_strength_v128/r2']], 4)}

Full model metrics: `outputs/hbond_analysis/hbond_length_controlled_model_ablation_metrics.csv`.

Main plots:

- `outputs/hbond_analysis/plots/hbond_length_controlled_model_ablation_r2.png`
- `outputs/hbond_analysis/plots/hbond_partial_correlation_after_length_control.png`

### Conclusion

Length normalization is useful, but it is not enough on its own. The raw correlation analysis overestimates hydrogen-bond importance because raw count features strongly encode protein size. After controlling for length, the remaining hydrogen-bond signal becomes more specific: density, nonlocality, contact order, and strong nonlocal hydrogen bonds are more informative than total hydrogen-bond count alone.

The model ablation is the most decision-relevant result. If `length + controlled hydrogen-bond features` improves over `length_only`, then hydrogen bonding is not merely a proxy for sequence length. In this dataset, the controlled hydrogen-bond descriptors should be treated as interpretable structural features and can be tested as auxiliary inputs to the mechanical-property predictor or as physics-informed reward components.

For the next modeling step, prefer topology-aware and residualized hydrogen-bond descriptors rather than raw hydrogen-bond counts. In particular, `nonlocal_hbond_per_residue`, `hbond_contact_order`, `nonlocal_backbone_backbone_fraction`, and `strong_nonlocal_fraction` are more chemically meaningful candidates than `hbond_count` alone.
"""

start_marker = "\n## Length-Controlled Hydrogen-Bond Analysis\n"
old = MD_PATH.read_text(encoding="utf-8")
if start_marker in old:
    old = old.split(start_marker)[0].rstrip() + "\n"
MD_PATH.write_text(old + section, encoding="utf-8")

summary = {
    "partial_correlation_table": str(BASE_DIR / "hbond_length_controlled_partial_correlations.csv"),
    "stratified_correlation_table": str(BASE_DIR / "hbond_length_stratified_correlations.csv"),
    "matched_pairs": str(BASE_DIR / "hbond_matched_length_pairs.csv"),
    "matched_pair_summary": str(BASE_DIR / "hbond_matched_length_pair_summary.csv"),
    "model_ablation_metrics": str(BASE_DIR / "hbond_length_controlled_model_ablation_metrics.csv"),
    "model_ablation_test_deltas": str(BASE_DIR / "hbond_length_controlled_model_ablation_test_deltas.csv"),
}
(BASE_DIR / "hbond_length_disentanglement_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Updated", MD_PATH)
summary

Updated hbond_analysis.md


{'partial_correlation_table': 'outputs/hbond_analysis/hbond_length_controlled_partial_correlations.csv',
 'stratified_correlation_table': 'outputs/hbond_analysis/hbond_length_stratified_correlations.csv',
 'matched_pairs': 'outputs/hbond_analysis/hbond_matched_length_pairs.csv',
 'matched_pair_summary': 'outputs/hbond_analysis/hbond_matched_length_pair_summary.csv',
 'model_ablation_metrics': 'outputs/hbond_analysis/hbond_length_controlled_model_ablation_metrics.csv',
 'model_ablation_test_deltas': 'outputs/hbond_analysis/hbond_length_controlled_model_ablation_test_deltas.csv'}

In [10]:
!conda install -c conda-forge tabulate

Solving environment: done


==> WARNING: A newer version of conda exists. <==
  current version: 25.7.0
  latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c defaults conda

Or to minimize the number of packages updated during conda update use

     conda install conda=26.5.3



# All requested packages already installed.

